# Computer Vision Passenger Counter: Step-by-Step Explanation

This notebook demonstrates the methodology for counting passengers and detecting footboard violations using **YOLOv8** and **ByteTrack**.

## 1. Setup & Load Model
We use **YOLOv8 Nano** for fast, real-time object detection, limiting the detection class to `0` (Person).

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import supervision as sv
import matplotlib.pyplot as plt

# Load the YOLOv8 model
model = YOLO('yolov8n.pt')
print("YOLOv8 model loaded successfully.")

## 2. Define Tracking & Virtual Zones
We use **ByteTrack** to maintain consistent IDs across frames. We also define:
1. **Line Zone**: To count passengers crossing in and out.
2. **Polygon Zone**: To detect passengers lingering on the footboard.

In [ ]:
# Initialize Tracker
tracker = sv.ByteTrack()

# Define Dummy Resolution
width, height = 640, 480

# 1. Line Zone (Counting)
START = sv.Point(0, height // 2)
END = sv.Point(width, height // 2)
line_zone = sv.LineZone(start=START, end=END)

# 2. Footboard Zone (Bottom 30%)
footboard_polygon = np.array([
    [0, int(height * 0.7)],
    [width, int(height * 0.7)],
    [width, height],
    [0, height]
])
footboard_zone = sv.PolygonZone(polygon=footboard_polygon, frame_resolution_wh=(width, height))
print("Zones initialized.")

## 3. Simulating a Frame Processing Step
Let's see how a single frame is processed to detect, track, and annotate passengers.

In [ ]:
def process_frame(frame_path):
    # Note: For the actual presentation, provide a real image file
    frame = cv2.imread(frame_path)
    if frame is None:
        print(f"Could not load {frame_path}. Creating a dummy frame.")
        frame = np.zeros((480, 640, 3), dtype=np.uint8)
    
    # 1. Inference
    result = model(frame, classes=[0], verbose=False)[0]
    detections = sv.Detections.from_ultralytics(result)
    
    # 2. Tracking
    detections = tracker.update_with_detections(detections)
    
    # 3. Annotation Prep
    box_annotator = sv.BoxAnnotator(thickness=2)
    label_annotator = sv.LabelAnnotator()
    
    if hasattr(detections, 'confidence') and detections.confidence is not None:
        labels = [f"ID:{t_id} | Conf:{conf:.2f}" for t_id, conf in zip(detections.tracker_id, detections.confidence)]
    else:
        labels = ["Person" for _ in range(len(detections))]
        
    # Annotate
    annotated_frame = box_annotator.annotate(scene=frame.copy(), detections=detections)
    if len(detections) > 0:
        annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)
        
    # Show Result
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
    plt.title("Processed Frame with YOLOv8 & ByteTrack")
    plt.axis('off')
    plt.show()

print("Ready to process frames! Run: process_frame('sample.jpg')")